# Senate meeting minutes processor
Last updated: _2026.10.01_

## Configuration

In [1]:
WORK_DIR = "senate_meeting_minutes"
INPUT_URL = "https://www.uni-corvinus.hu/fooldal/egyetemunkrol/szenatus-egyetemi-testuletek/szenatus/"

## URL crawler

In [2]:
import re

import requests

import pandas as pd

from pathlib import Path
from getpass import getpass
from bs4 import BeautifulSoup

from urllib.parse import unquote, urljoin, urlparse

In [3]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(headers)

protected_page = session.get(INPUT_URL, timeout=30)
protected_soup = BeautifulSoup(protected_page.text, "html.parser")
login_link = protected_soup.find(
    "a",
    href=re.compile(r"wp-login\.php(?:\?|$)"),
)

if (
    protected_page.status_code < 400
    and protected_soup.select_one(".sidebar-nav") is not None
):
    authenticated_page = protected_page
    authenticated_soup = protected_soup
elif login_link is None:
    protected_page.raise_for_status()
    raise RuntimeError(
        f"Could not find .sidebar-nav or a login link at {protected_page.url!r}."
    )
else:
    login_page = session.get(
        urljoin(protected_page.url, login_link["href"]),
        timeout=30,
    )
    login_soup = BeautifulSoup(login_page.text, "html.parser")
    login_form = login_soup.find("form")

    if login_page.status_code >= 400 and login_page.status_code != 401:
        login_page.raise_for_status()
    if login_form is None:
        if login_page.status_code == 401:
            raise RuntimeError(
                f"The SSO login page returned 401 Unauthorized at {login_page.url!r}; "
                "the login flow cannot continue because no login form was provided."
            )
        raise RuntimeError(f"Could not find the login form at {login_page.url!r}.")

    hidden_fields = {
        field["name"]: field.get("value", "")
        for field in login_form.select("input[type=hidden][name]")
    }

    username = input("Corvinus username: ").strip()
    if not username:
        raise ValueError("A Corvinus username is required.")

    password = getpass("Corvinus password: ")
    if not password:
        raise ValueError("A Corvinus password is required.")

    login_data = {
        **hidden_fields,
        "username": username,
        "password": password,
    }
    login_data.setdefault("_eventId", "submit")

    login_action = urljoin(
        login_page.url,
        login_form.get("action") or login_page.url,
    )
    authenticated_page = session.post(
        login_action,
        data=login_data,
        timeout=30,
    )
    authenticated_soup = BeautifulSoup(authenticated_page.text, "html.parser")

    if authenticated_page.status_code == 401 or authenticated_page.url.startswith(
        "https://sso.uni-corvinus.hu/cas/login"
    ):
        error = authenticated_soup.select_one("#msg, .errors, .alert-danger")
        detail = error.get_text(" ", strip=True) if error else "credentials were rejected"
        raise RuntimeError(f"CAS login failed: {detail}")

    authenticated_page.raise_for_status()

Corvinus username:  bzoltan1
Corvinus password:  ··············


In [4]:
output_dir = Path.cwd() / WORK_DIR
output_dir.mkdir(parents=True, exist_ok=True)

sidebar_links = []
seen_sidebar_urls = set()
for link in authenticated_soup.select(".sidebar-nav a[href]"):
    sidebar_url = urljoin(authenticated_page.url, link["href"])
    if sidebar_url in seen_sidebar_urls:
        continue

    seen_sidebar_urls.add(sidebar_url)
    sidebar_label = link.get_text(" ", strip=True)
    if not sidebar_label:
        sidebar_label = link.get("aria-label") or link.get("title") or sidebar_url

    sidebar_links.append(
        {
            "label": sidebar_label,
            "url": sidebar_url,
        }
    )

if not sidebar_links:
    raise RuntimeError(
        "No links were found in .sidebar-nav after login; the page may have changed "
        "or the account may not have access."
    )

pd.DataFrame(sidebar_links, columns=["label", "url"]).to_csv(
    output_dir / "senate_sidebar_links.csv",
    index=False,
    encoding="utf-8-sig",
)

In [5]:
pdf_records = []
not_found_links = []
unauthorized_links = []
connection_error_links = []
seen_pdf_urls = set()
for page_index, sidebar_link in enumerate(sidebar_links, start=1):
    print(
        f"Reading {page_index}/{len(sidebar_links)}: "
        f"{sidebar_link['label']}"
    )
    try:
        section_page = session.get(sidebar_link["url"], timeout=30)
    except requests.exceptions.ConnectionError as error:
        print(f"  Connection error: {sidebar_link['url']} ({error})")
        connection_error_links.append(
            {
                "type": "sidebar_page",
                "label": sidebar_link["label"],
                "url": sidebar_link["url"],
                "error": str(error),
            }
        )
        continue

    if section_page.status_code == 401:
        print(f"  401 Unauthorized: {section_page.url}")
        unauthorized_links.append(
            {
                "type": "sidebar_page",
                "label": sidebar_link["label"],
                "url": sidebar_link["url"],
                "error": "Unauthorized",
            }
        )
        continue

    if section_page.status_code == 404:
        print(f"  404 Not Found: {section_page.url}")
        not_found_links.append(
            {
                "type": "sidebar_page",
                "label": sidebar_link["label"],
                "url": sidebar_link["url"],
            }
        )
        continue

    section_page.raise_for_status()
    section_soup = BeautifulSoup(section_page.text, "html.parser")
    main = section_soup.find("main")

    if main is None:
        print(f"  No <main> tag found at {section_page.url}")
        continue

    for pdf_link in main.select("a[href]"):
        pdf_url = urljoin(section_page.url, pdf_link["href"])
        if not urlparse(pdf_url).path.lower().endswith(".pdf"):
            continue
        if pdf_url in seen_pdf_urls:
            continue

        seen_pdf_urls.add(pdf_url)
        pdf_label = pdf_link.get_text(" ", strip=True)
        if not pdf_label:
            pdf_label = pdf_link.get("aria-label") or pdf_link.get("title") or pdf_url

        pdf_records.append(
            {
                "sidebar_label": sidebar_link["label"],
                "sidebar_url": sidebar_link["url"],
                "pdf_label": pdf_label,
                "pdf_url": pdf_url,
                "status": "pending",
                "error": "",
            }
        )

if (
    not pdf_records
    and not not_found_links
    and not unauthorized_links
    and not connection_error_links
):
    raise RuntimeError(
        "No PDF links were found in the <main> tag of the sidebar pages; "
        "the page structure or account access may have changed."
    )

Reading 1/19: Szenátus
Reading 2/19: A Szenátus tagjai
Reading 3/19: 2026/2027
Reading 4/19: 2025/2026
Reading 5/19: 2024/2025
Reading 6/19: 2023/2024
Reading 7/19: 2022/2023
Reading 8/19: 2021/2022
Reading 9/19: 2020/2021
Reading 10/19: 2019/2020
Reading 11/19: 2018/2019
Reading 12/19: 2017/2018
Reading 13/19: 2016/2017
Reading 14/19: 2015/2016
Reading 15/19: 2014/2015
Reading 16/19: 2013/2014
Reading 17/19: 2012/2013
Reading 18/19: 2011/2012
Reading 19/19: 2010/2011


In [6]:
pdf_records[:5]

[{'sidebar_label': 'Szenátus',
  'sidebar_url': 'https://www.uni-corvinus.hu/fooldal/egyetemunkrol/szenatus-egyetemi-testuletek/szenatus/',
  'pdf_label': 'Adatkezelési tájékoztató',
  'pdf_url': 'https://www.uni-corvinus.hu/contents/uploads/2024/05/AT_Szenátus_240515.bc3.pdf',
  'status': 'pending',
  'error': ''},
 {'sidebar_label': '2026/2027',
  'sidebar_url': 'https://www.uni-corvinus.hu/fooldal/egyetemunkrol/szenatus-egyetemi-testuletek/szenatus/2026-2027/',
  'pdf_label': 'Írásbeli szavazás elrendelése',
  'pdf_url': 'https://www.uni-corvinus.hu/downloads/di0f.o8kflc/bce-sz-20260928-irasbeli-szavazas-elrendelese-sgd.pdf',
  'status': 'pending',
  'error': ''},
 {'sidebar_label': '2026/2027',
  'sidebar_url': 'https://www.uni-corvinus.hu/fooldal/egyetemunkrol/szenatus-egyetemi-testuletek/szenatus/2026-2027/',
  'pdf_label': 'Meghívó',
  'pdf_url': 'https://www.uni-corvinus.hu/downloads/dgpo.ok9t7o/senate-invitation-260922-sgd.pdf',
  'status': 'pending',
  'error': ''},
 {'sideba

In [7]:
used_filenames = set()
for index, record in enumerate(pdf_records, start=1):
    pdf_url = record["pdf_url"]
    filename = Path(unquote(urlparse(pdf_url).path)).name
    filename = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", filename)
    if not filename:
        filename = f"senate_{index:04d}.pdf"

    original_filename = filename
    duplicate_number = 2
    while filename in used_filenames:
        path = Path(original_filename)
        filename = f"{path.stem}_{duplicate_number}{path.suffix}"
        duplicate_number += 1
    used_filenames.add(filename)
    record["filename"] = filename

    destination = output_dir / filename
    partial_destination = destination.with_suffix(destination.suffix + ".part")
    print(f"Downloading {index}/{len(pdf_records)}: {filename}")

    try:
        with session.get(pdf_url, stream=True, timeout=60) as pdf_response:
            if pdf_response.status_code == 401:
                record["status"] = "401"
                record["error"] = "Unauthorized"
                unauthorized_links.append(
                    {
                        "type": "pdf",
                        "label": record["pdf_label"],
                        "url": pdf_url,
                        "error": "Unauthorized",
                    }
                )
                print(f"  401 Unauthorized: {pdf_url}")
                continue

            if pdf_response.status_code == 404:
                record["status"] = "404"
                record["error"] = "Not Found"
                not_found_links.append(
                    {
                        "type": "pdf",
                        "label": record["pdf_label"],
                        "url": pdf_url,
                    }
                )
                print(f"  404 Not Found: {pdf_url}")
                continue

            pdf_response.raise_for_status()
            first_chunk = next(pdf_response.iter_content(chunk_size=8192), b"")
            if not first_chunk.startswith(b"%PDF"):
                raise RuntimeError(
                    f"The response for {pdf_url} was not a PDF; "
                    "the authenticated session may have expired."
                )

            with partial_destination.open("wb") as output_file:
                output_file.write(first_chunk)
                for chunk in pdf_response.iter_content(chunk_size=8192):
                    if chunk:
                        output_file.write(chunk)

        partial_destination.replace(destination)
        record["status"] = "downloaded"
    except requests.exceptions.ConnectionError as error:
        record["status"] = "connection_error"
        record["error"] = str(error)
        connection_error_links.append(
            {
                "type": "pdf",
                "label": record["pdf_label"],
                "url": pdf_url,
                "error": str(error),
            }
        )
        print(f"  Connection error: {pdf_url} ({error})")
    finally:
        try:
            partial_destination.unlink()
        except FileNotFoundError:
            pass

  404 Not Found: https://www.uni-corvinus.hu/contents/uploads/2022/04/SZ_jegyzokonyv_220316.b06.pdf
  404 Not Found: https://www.uni-corvinus.hu/contents/uploads/2021/07/hatarozatok_210622.08e.pdf
  404 Not Found: https://www.uni-corvinus.hu/contents/uploads/2021/06/1_EB_IFT.61c.pdf
  404 Not Found: https://www.uni-corvinus.hu/contents/uploads/2021/06/2_SZMR.af8.pdf
  404 Not Found: https://www.uni-corvinus.hu/contents/uploads/2021/06/hatarozatok_210518.12c.pdf
  404 Not Found: https://www.uni-corvinus.hu/contents/uploads/2021/05/4_1_beszamolo.974.pdf
  404 Not Found: https://www.uni-corvinus.hu/contents/uploads/2021/05/4_2_konyvvizsgaloi_jelentes.ee8.pdf
  404 Not Found: https://www.uni-corvinus.hu/contents/uploads/2021/05/4_3_KFB_jelent%C3%A9s.730.pdf
  Connection error: http://www.uni-corvinus.hu/fileadmin/user_upload/cornet/hu/kozponti_szervezeti_egysegek/rektori_hivatal/files/szenatus/2017_2018/SZ_171219/3a_2018_elemi_koltsegvetes.pdf (HTTPConnectionPool(host='old.uni-corvinus.hu'

In [8]:
manifest_columns = [
    "sidebar_label",
    "sidebar_url",
    "pdf_label",
    "pdf_url",
    "filename",
    "status",
    "error",
]
pd.DataFrame(pdf_records, columns=manifest_columns).to_csv(
    output_dir / "senate_pdf_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)
pd.DataFrame(not_found_links, columns=["type", "label", "url"]).to_csv(
    output_dir / "senate_404_links.csv",
    index=False,
    encoding="utf-8-sig",
)
pd.DataFrame(
    unauthorized_links,
    columns=["type", "label", "url", "error"],
).to_csv(
    output_dir / "senate_401_links.csv",
    index=False,
    encoding="utf-8-sig",
)
pd.DataFrame(
    connection_error_links,
    columns=["type", "label", "url", "error"],
).to_csv(
    output_dir / "senate_connection_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

downloaded_count = sum(record["status"] == "downloaded" for record in pdf_records)
print(f"Downloaded {downloaded_count} of {len(pdf_records)} PDFs to {output_dir}")
print(f"Saved PDF metadata to {output_dir / 'senate_pdf_manifest.csv'}")
if not_found_links:
    print(
        f"Skipped {len(not_found_links)} links that returned 404; "
        f"details saved to {output_dir / 'senate_404_links.csv'}"
    )
if unauthorized_links:
    print(
        f"Skipped {len(unauthorized_links)} links that returned 401; "
        f"details saved to {output_dir / 'senate_401_links.csv'}"
    )
if connection_error_links:
    print(
        f"Skipped {len(connection_error_links)} links due to connection errors; "
        f"details saved to {output_dir / 'senate_connection_errors.csv'}"
    )

Downloaded 445 of 454 PDFs to D:\Work\Jacob\senate_meeting_minutes
Saved PDF metadata to D:\Work\Jacob\senate_meeting_minutes\senate_pdf_manifest.csv
Skipped 8 links that returned 404; details saved to D:\Work\Jacob\senate_meeting_minutes\senate_404_links.csv
Skipped 1 links due to connection errors; details saved to D:\Work\Jacob\senate_meeting_minutes\senate_connection_errors.csv


In [9]:
pdf_files = pd.DataFrame(pdf_records, columns=manifest_columns).rename(columns={
    'pdf_label': 'label',
    'pdf_url': 'url'
})

In [10]:
pdf_files[:5]

,sidebar_label,sidebar_url,label,url,filename,status,error
0,Szenátus,https://www.uni-corvinus.hu/fooldal/egyetemunk...,Adatkezelési tájékoztató,https://www.uni-corvinus.hu/contents/uploads/2...,AT_Szenátus_240515.bc3.pdf,downloaded,
1,2026/2027,https://www.uni-corvinus.hu/fooldal/egyetemunk...,Írásbeli szavazás elrendelése,https://www.uni-corvinus.hu/downloads/di0f.o8k...,bce-sz-20260928-irasbeli-szavazas-elrendelese-...,downloaded,
2,2026/2027,https://www.uni-corvinus.hu/fooldal/egyetemunk...,Meghívó,https://www.uni-corvinus.hu/downloads/dgpo.ok9...,senate-invitation-260922-sgd.pdf,downloaded,
3,2026/2027,https://www.uni-corvinus.hu/fooldal/egyetemunk...,1. melléklet,https://www.uni-corvinus.hu/downloads/dgqg.m0l...,1-local-resilience.pdf,downloaded,
4,2026/2027,https://www.uni-corvinus.hu/fooldal/egyetemunk...,2. melléklet,https://www.uni-corvinus.hu/downloads/dgqh.16d...,2-nedimarc-neuro-and-digital-marketing.pdf,downloaded,


In [11]:
print("Number of meeting minutes downloaded: {:,}".format(
    pdf_files[pdf_files['label'].str.contains('jegyz', case=False, regex=False)].shape[0]
))

Number of meeting minutes downloaded: 59


In [12]:
pdf_files[pdf_files['label'].str.contains('jegyz', case=False, regex=False)]['status'].value_counts()

status
downloaded    58
404            1
Name: count, dtype: int64

In [13]:
pdf_files[pdf_files['label'].str.contains('jegyz', case=False, regex=False)]['label'].unique()

array(['Jegyzőkönyv', 'Egyetemi Doktori Tanács jegyzőkönyvi kivonata',
       'jegyzokonyv_200512.3b3.pdf'], dtype=object)

In [14]:
pdf_files[pdf_files['label'].str.contains('jegyz', case=False, regex=False)][:5]

,sidebar_label,sidebar_url,label,url,filename,status,error
22,2025/2026,https://www.uni-corvinus.hu/fooldal/egyetemunk...,Jegyzőkönyv,https://www.uni-corvinus.hu/downloads/d4wm.15x...,jegyzokonyv-260603.pdf,downloaded,
37,2025/2026,https://www.uni-corvinus.hu/fooldal/egyetemunk...,Jegyzőkönyv,https://www.uni-corvinus.hu/downloads/cwdy.1ru...,jegyzokonyv-260409.pdf,downloaded,
41,2025/2026,https://www.uni-corvinus.hu/fooldal/egyetemunk...,Jegyzőkönyv,https://www.uni-corvinus.hu/downloads/cv8x.ap3...,jegyzokonyv-260330.pdf,downloaded,
44,2025/2026,https://www.uni-corvinus.hu/fooldal/egyetemunk...,Jegyzőkönyv,https://www.uni-corvinus.hu/downloads/cxp1.bu1...,sz-jegyzokonyv-250317.pdf,downloaded,
48,2025/2026,https://www.uni-corvinus.hu/fooldal/egyetemunk...,Jegyzőkönyv,https://www.uni-corvinus.hu/downloads/dfvm.13b...,sz-jegyzokonyv-260303-ai.pdf,downloaded,


In [15]:
pdf_files = pdf_files[pdf_files['label'].str.contains('jegyz', case=False, regex=False)].copy()

## OCRing the PDFs

In [16]:
import re
import json
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd

from PIL import Image

In [17]:
OCR_LANGUAGE = "hun"
OCR_DPI = 300

In [18]:
def extract_pdf_text(pdf_path: Path) -> str:
    """
    Extract text from a PDF.

    First tries normal PDF text extraction.
    If the PDF is scanned/image-based, falls back to OCR.
    """

    print(f"Processing: {pdf_path.name}")

    with fitz.open(pdf_path) as doc:

        # -----------------------------------------------------------
        # 1. Try native PDF text extraction
        # -----------------------------------------------------------

        native_text = "\n\n".join(
            page.get_text("text")
            for page in doc
        ).strip()

        # If enough text was extracted, use it
        if len(native_text) > 100:
            print("  -> Native text extraction")
            return native_text

        # -----------------------------------------------------------
        # 2. OCR fallback
        # -----------------------------------------------------------

        print("  -> Scanned PDF detected, using OCR")

        ocr_pages = []

        zoom = OCR_DPI / 72
        matrix = fitz.Matrix(zoom, zoom)

        for page_number, page in enumerate(doc, start=1):

            print(f"     OCR page {page_number}/{len(doc)}")

            # Render page to image
            pix = page.get_pixmap(
                matrix=matrix,
                alpha=False
            )

            image = Image.frombytes(
                "RGB",
                [pix.width, pix.height],
                pix.samples
            )

            # OCR
            text = pytesseract.image_to_string(
                image,
                lang=OCR_LANGUAGE,
                config="--psm 3"
            )

            ocr_pages.append(text.strip())

        return "\n\n".join(ocr_pages)


In [19]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print(pytesseract.get_languages())

['eng', 'hun', 'osd']


In [21]:
pdf_files['filename'][:5]

22          jegyzokonyv-260603.pdf
37          jegyzokonyv-260409.pdf
41          jegyzokonyv-260330.pdf
44       sz-jegyzokonyv-250317.pdf
48    sz-jegyzokonyv-260303-ai.pdf
Name: filename, dtype: object

In [24]:
if not Path(WORK_DIR).exists():
    raise FileNotFoundError(
        f"Output directory does not exist: {WORK_DIR}"
    )

files_to_ocr = []
for f in pdf_files['filename'].values:
    if Path(WORK_DIR + "/" + f).exists():
        files_to_ocr.append(Path(WORK_DIR + "/" + f))
    else:
        print("Missing: {}".format(WORK_DIR + "/" + f))

print(f"Found {len(files_to_ocr)} PDF files\n")

Missing: senate_meeting_minutes/SZ_jegyzokonyv_220316.b06.pdf
Found 58 PDF files



In [25]:
# ---------------------------------------------------------
# Checkpoint configuration
# ---------------------------------------------------------

CHECKPOINT_FILE = Path(WORK_DIR) / "extraction_checkpoint.json"

# Load existing checkpoint if available
if CHECKPOINT_FILE.exists():
    try:
        with CHECKPOINT_FILE.open("r", encoding="utf-8") as f:
            records = json.load(f)

        print(
            f"Loaded checkpoint: {len(records)} PDFs already processed\n"
        )

    except (json.JSONDecodeError, OSError) as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting with an empty checkpoint.\n")
        records = []
else:
    records = []


# Keep track of files already successfully processed
processed_files = {
    record["filename"]
    for record in records
}


# ---------------------------------------------------------
# Process PDFs
# ---------------------------------------------------------

for pdf_path in files_to_ocr:
    # Skip files already successfully processed
    if pdf_path.name in processed_files:
        print(f"SKIP: {pdf_path.name}")
        continue

    print(f"Processing: {pdf_path.name}")

    try:
        text = extract_pdf_text(pdf_path)

        # Extract year from filename, e.g.
        # 2020_some_document.pdf -> 2020
        year_match = re.match(
            r"^(\d{4})_",
            pdf_path.name
        )

        year = (
            int(year_match.group(1))
            if year_match
            else None
        )

        record = {
            "year": year,
            "filename": pdf_path.name,
            "text": text,
        }

        records.append(record)

        # -------------------------------------------------
        # Save checkpoint immediately after each PDF
        # -------------------------------------------------

        with CHECKPOINT_FILE.open(
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                records,
                f,
                ensure_ascii=False,
                indent=2
            )

        processed_files.add(pdf_path.name)

        print(
            f"  Extracted {len(text):,} characters"
        )
        print(
            f"  Checkpoint saved ({len(records)} PDFs)\n"
        )

    except Exception as e:
        print(
            f"ERROR processing {pdf_path.name}: {e}\n"
        )

Processing: jegyzokonyv-260603.pdf
Processing: jegyzokonyv-260603.pdf
  -> Native text extraction
  Extracted 1,910 characters
  Checkpoint saved (1 PDFs)

Processing: jegyzokonyv-260409.pdf
Processing: jegyzokonyv-260409.pdf
  -> Native text extraction
  Extracted 1,868 characters
  Checkpoint saved (2 PDFs)

Processing: jegyzokonyv-260330.pdf
Processing: jegyzokonyv-260330.pdf
  -> Native text extraction
  Extracted 3,569 characters
  Checkpoint saved (3 PDFs)

Processing: sz-jegyzokonyv-250317.pdf
Processing: sz-jegyzokonyv-250317.pdf
  -> Native text extraction
  Extracted 34,889 characters
  Checkpoint saved (4 PDFs)

Processing: sz-jegyzokonyv-260303-ai.pdf
Processing: sz-jegyzokonyv-260303-ai.pdf
  -> Scanned PDF detected, using OCR
     OCR page 1/22
     OCR page 2/22
     OCR page 3/22
     OCR page 4/22
     OCR page 5/22
     OCR page 6/22
     OCR page 7/22
     OCR page 8/22
     OCR page 9/22
     OCR page 10/22
     OCR page 11/22
     OCR page 12/22
     OCR page 13/22

In [26]:
from Helper import Log

In [27]:
df = pd.DataFrame(records)

In [28]:
import statistics

print("Average length of OCRd text: {:,.2f}".format(
    statistics.mean([i for i in df['text'].str.len()])
))

Average length of OCRd text: 63,629.76


In [29]:
with Log.save_file(WORK_DIR + "/output/senate_meeting_minutes_hun.csv") as filename:
    df.to_csv(filename, index=False)

File saved to `senate_meeting_minutes\output\senate_meeting_minutes_hun_202610011325.csv` (3.7MB)


In [33]:
# Not a meeting minute
print(df.loc[0, 'text'])

SZENÁTUS 
Ülésen kívüli szavazás 
küldi: 
Jog és Beszerzés 
e-mail: 
szenatus@uni-corvinus.hu 
dátum: 
2026. június 4. 
 
 
 
JEGYZŐKÖNYV 
a Szenátus ülésen kívüli szavazásának lezárásáról 
 
SZ-XV/2025/2026. (VI. 3.) 
 
Bruno Joseph T. van Pottelsberghe de la Potterie, elnök-rektor a Szervezeti és Működési 
Rend 10. § (16) bekezdése alapján a Szenátus összehívása helyett ülésen kívül történő 
szavazást rendelt el az alábbi témában: 
 
1. Egyetemi Doktori Szabályzat módosításának jóváhagyása 
Előterjesztő: Előterjesztő: Bartus Tamás, kutatási és akadémiai közösségért felelős 
rektorhelyettes 
 
Az írásbeli szavazás elrendelésének időpontja: 2026. május 29. 
A szavazati jog gyakorlásának időtartama:  
2026. június 3. (szerda) 8:00 óra – 2026. június 3. (szerda) 16:30 óra  
 
A szavazatszámláló bizottság tagjai:  
- 
Kónya István Tamás, a Szenátus tagja 
- 
K. Szabó Áron, a Szenátus tagja 
- 
Toronyai Gábor, a Szenátus tagja 
 
A szavazatszámláló bizottság Teams értekezlet keretében tart

In [36]:
# Meeting minute
print(df.loc[3, 'text'])

————— 
1 | 10 
 
Bruno van Pottelsberghe: Jó napot kívánok, welcome to all of you, to all the Senate 
members and the guests. 
We will start with the certificate awards. You know that it has become a tradition at 
Corvinus for years to celebrate the academic advancement of our colleagues, our 
newly joined faculty and researchers, as well as the Univeristy’s new Professors 
Emeriti. And so we have a long series of colleagues who have been appointed or 
promoted and I think it's important for us to celebrate this together because they join 
our community. And so we will start this Senate by giving them the certificate and a 
small gift. And if you allow me, I will stand because today we have many of them. 
And so I start with Róbert Imre Csehi, Institute of Social and Political Science, 
Associate Professor. Nóra Ágota Felföldi-Szűcs, Associate Professor, Institute of 
Finance. Assistant Professor, Institute of Marketing and Communication Sciences, 
Evelin Horváth. Assistant Professor, 

__Refined prompt__

You are an expert in analysing institutional meeting transcripts, especially Senate, board, committee, and academic governance meetings.

Your task is to analyse the supplied transcript and extract structured information about:

1. conversations and substantive discussions;
2. voting;
3. discussion segments;
4. concepts discussed;
5. actors participating in the discussions;
6. each actor's attitude toward each concept;
7. evidence supporting each attitude;
8. confidence in each extracted relationship;
9. an actor × concept attitude matrix summarising the results.

The analysis must be based exclusively on the supplied transcript. Do not use external knowledge or assumptions about the people, organisations, political positions, or events mentioned in the transcript.

---

# 1. Conversation detection

Set **`contains_conversation`** to **`1`** only when the text contains an actual substantive exchange between two or more actors.

Examples include:

- questions and answers;
- comments followed by responses;
- disagreement or agreement;
- discussion of a proposal;
- arguments and counterarguments;
- participants challenging or clarifying another participant's statement;
- participants expressing different views on the same issue.

Set **`contains_conversation`** to **`0`** if there is no substantive exchange.

Do NOT treat the following by themselves as a conversation:

- a speech or presentation by one person;
- reading an announcement;
- listing agenda items;
- procedural instructions;
- introductions;
- ceremonial remarks;
- simple acknowledgements such as "thank you";
- mentioning another person's opinion without engaging with it.

A meeting can contain voting without containing substantive conversation.

---

# 2. Voting detection

Set **`contains_voting`** to **`1`** when the text contains an actual or explicitly initiated voting procedure.

Examples include:

- a request to vote;
- a vote on an agenda item;
- a recorded vote or voting result;
- a motion being put to a vote;
- approval or rejection through voting;
- written or electronic voting;
- yes/no voting;
- abstentions;
- an explicit reference to a previous written vote.

Set **`contains_voting`** to **`0`** when voting is not present.

Do not infer voting merely because a decision, approval, adoption, or rejection is mentioned.

There must be textual evidence that voting occurred, was requested, or was explicitly referred to as voting.

---

# 3. Discussion segments

Divide the transcript into meaningful **`discussion_segments`**.

A discussion segment is a contiguous portion of the transcript dealing with one substantive issue, concept, proposal, question, or disagreement.

Do NOT create one segment for every speaker turn.

If several speakers discuss the same issue consecutively, group their contributions into one segment.

Start a new segment when the discussion clearly changes to another substantive issue.

For every segment provide:

- a unique **`segment_id`**;
- a short title;
- the approximate location in the transcript;
- the main topic;
- a concise summary;
- the actors participating;
- the concepts discussed.

Example:

{
"segment_id": "S3",
"title": "Postponement of the V4 conference",
"location": "Pages 4-10",
"summary": "Participants discuss the postponement of the conference, political sensitivity, academic freedom, institutional neutrality, and the role of university management.",
"actor_ids": ["A1", "A2", "A3"],
"concept_ids": ["C1", "C2", "C3"]
}

---

# 4. Concepts

A concept is a substantive issue, proposal, principle, decision, event, or topic that participants discuss.

Do not create a separate concept for every sentence.

Group semantically related statements into a single concept.

Prefer concise, meaningful concept names such as:

- "Postponement of the conference"
- "Academic freedom"
- "Political neutrality"
- "Political sensitivity of the research"
- "Scientific integrity"
- "Institutional reputational risk"
- "Political interference"
- "University censorship"

Do not automatically treat every noun mentioned in the transcript as a concept.

A concept should be something that participants actually discuss, evaluate, question, support, reject, or otherwise engage with.

If the same concept appears in multiple discussion segments, normally create only one concept and link it to all relevant segments.

Create separate concepts only when the issues are substantively distinct.

---

# 5. Actors

An actor is a person or clearly identifiable group that meaningfully participates in a discussion.

Use the actor's full name when available.

Examples:

- "Bruno van Pottelsberghe"
- "Eszter Kirs"
- "Kata Apáti-Tóth"
- "Mánuel László Mágó"
- "The Senate"

Do not include people who are merely mentioned but do not participate meaningfully.

Distinguish individuals from groups or institutions when the transcript supports this distinction.

Every actor receives a unique **`actor_id`**.

---

# 6. Actor-concept relationships

For every actor who meaningfully addresses a concept, create an **`actor_attitude`** relationship.

An actor does NOT need to have a relationship with every concept in a discussion segment.

Only create a relationship when the transcript contains evidence that the actor actually addressed the concept.

Each relationship must contain:

- actor;
- concept;
- attitude;
- confidence;
- evidence;
- discussion segment(s).

---

# 7. Attitude classification

Classify the actor's expressed attitude toward each concept using exactly one of:

### `"positive"`

The actor explicitly:

- supports;
- approves;
- agrees with;
- endorses;
- defends;
- favors;

the concept.

### `"negative"`

The actor explicitly:

- opposes;
- criticizes;
- rejects;
- objects to;
- condemns;
- expresses an unfavorable view;

toward the concept.

### `"neutral"`

The actor discusses, describes, explains, questions, or refers to the concept without expressing a clearly positive or negative attitude.

### `"mixed"`

The actor expresses both favorable and unfavorable views toward the same concept.

For example:

"I understand why the conference was postponed, but I strongly disagree with postponing it."

This is **`"mixed"`** toward the concept of postponement.

### `"unclear"`

There is insufficient evidence to reliably determine the actor's attitude.

Do not guess.

---

# 8. Important distinction: attitude vs factual statement

Do not classify an actor as positive or negative simply because they make a factual claim.

For example:

"The conference was postponed."

→ **`"neutral"`**

"The postponement was completely unnecessary."

→ **`"negative"`**

"I believe postponing the conference protects the university."

→ **`"positive"`**

Similarly:

"The university has academic freedom."

is not automatically positive toward the concept of academic freedom.

The actor must express an evaluative position for the attitude to be positive or negative.

---

# 9. Important distinction: object of the attitude

Be precise about WHAT the actor is expressing an attitude toward.

For example, an actor might:

- support academic freedom;
- oppose postponing a conference;
- support postponing the conference;
- oppose political interference;
- criticize the university's reasoning for postponing the conference.

These are different relationships.

Do not collapse them into a general statement such as:

"Actor X is positive about the conference."

Identify the actual concept being evaluated.

---

# 10. Complex and conditional positions

When an actor expresses a nuanced or conditional position, analyse the individual concepts separately.

Example:

"I agree that the topic is politically sensitive, but I don't agree that this justifies postponing the event."

Possible extraction:

- Political sensitivity → positive/agreeing
- Postponement of the event → negative

Do not automatically classify the entire speaker's contribution as simply positive or negative.

---

# 11. Evidence

Every actor-concept relationship MUST include evidence.

Evidence should:

- come exclusively from the supplied transcript;
- be a concise quotation or faithful paraphrase;
- directly support the assigned attitude;
- identify the relevant statement or argument.

Keep quotations short.

If several statements support the classification, provide a concise synthesis rather than reproducing a long passage.

Example:

{
"attitude": "negative",
"evidence": "Eszter describes the postponement as censorship based on political grounds and questions why an academic event should be postponed because of political considerations."
}

Do not invent evidence.

---

# 12. Confidence score

Assign a numerical **`confidence`** between **`0`** and **`1`** to every actor-concept relationship.

The confidence score represents how strongly the transcript supports the extracted relationship and attitude.

Use the following guidance:

- **`0.90–1.00`**: explicit and unambiguous statement;
- **`0.75–0.89`**: strong evidence, with minor interpretation required;
- **`0.50–0.74`**: plausible interpretation, but somewhat ambiguous;
- **`0.25–0.49`**: weak evidence;
- **`0.00–0.24`**: very little evidence.

Use **`"unclear"`** when evidence is insufficient rather than assigning an artificially high confidence score.

The confidence score measures confidence in the **extraction**, not whether the actor's statement is factually correct.

For example:

"I strongly oppose postponing the conference."

→ attitude: **`"negative"`**
→ confidence: approximately **`0.98`**

"I understand the concern, although I'm not sure postponement is necessary."

→ attitude: **`"mixed"`**
→ confidence: approximately **`0.85`**

---

# 13. Changes of position

If an actor's attitude toward a concept changes during the transcript, do NOT hide the change.

Create separate actor-concept relationships for the relevant discussion segments.

For example:

- S1: Actor supports proposal.
- S4: Actor later expresses concerns about proposal.

Represent this as separate segment-level observations and, where appropriate, an overall relationship marked **`"mixed"`**.

The output should preserve the evolution of the discussion.

---

# 14. Discussion-level extraction

For every discussion segment, identify:

- participating actors;
- concepts discussed;
- actor-concept attitudes;
- supporting evidence;
- confidence scores.

An actor can participate in a segment without expressing an attitude toward every concept in that segment.

Do not force missing relationships.

---

# 15. Actor × concept attitude matrix

In addition to the detailed evidence-based relationships, create an **`attitude_matrix`**.

The matrix must have:

- one row for each actor;
- one column for each concept;
- the corresponding attitude in each cell.

Use these values:

- **`positive`**
- **`negative`**
- **`neutral`**
- **`mixed`**
- **`unclear`**
- **`not_discussed`**

Use **`"not_discussed"`** when there is no evidence that the actor addressed the concept.

Do NOT use **`"neutral"`** simply because no attitude was found.

This distinction is important:

- **`"neutral"`** = the actor discussed the concept but did not express a positive or negative attitude.
- **`"not_discussed"`** = there is no evidence that the actor addressed the concept.

Example:

{
"attitude_matrix": {
"actors": [
{
"actor_id": "A1",
"actor": "Bruno van Pottelsberghe",
"attitudes": {
"C1": "positive",
"C2": "neutral",
"C3": "negative"
}
},
{
"actor_id": "A2",
"actor": "Eszter Kirs",
"attitudes": {
"C1": "negative",
"C2": "positive",
"C3": "not_discussed"
}
}
]
}
}

The matrix is a summary of the detailed actor-concept relationships.

Do not introduce information into the matrix that does not exist in the detailed evidence.

---

# 16. Matrix confidence

For every matrix cell that is not **`"not_discussed"`**, also provide the confidence score and the relevant discussion segment IDs.

Use a structured matrix representation rather than only a string value.

For example:

{
"actor_id": "A1",
"actor": "Bruno van Pottelsberghe",
"attitudes": {
"C1": {
"attitude": "positive",
"confidence": 0.94,
"segment_ids": ["S3"]
},
"C2": {
"attitude": "neutral",
"confidence": 0.88,
"segment_ids": ["S3", "S4"]
},
"C3": {
"attitude": "not_discussed",
"confidence": null,
"segment_ids": []
}
}
}

This makes the matrix traceable back to the original discussion.

---

# 17. Overall concept summary

For every concept, provide a concise summary of the positions expressed by the different actors.

Do not determine which actor is correct.

Do not rank actors.

Do not determine which position is preferable.

The purpose is to represent the discussion faithfully.

Example:

{
"concept_id": "C1",
"concept": "Postponement of the conference",
"summary_of_positions": "Some participants describe the postponement as a reasonable temporary measure to avoid political sensitivity, while others view it as inappropriate interference with an academic event."
}

When positions are disputed, attribute them to the relevant actors rather than presenting a disputed interpretation as an objective fact.

---

# 18. Voting details

If **`contains_voting`** is **`1`**, identify the voting events where possible.

For each voting event provide:

- **`vote_id`**;
- the subject being voted on;
- the relevant discussion segment;
- the stated proposal/resolution;
- any explicitly reported result;
- explicitly identified individual votes or abstentions;
- whether the result is known.

Do not invent vote counts.

For example:

{
"vote_id": "V1",
"subject": "Amendment of the Rules of Procedure of the Senate",
"segment_id": "S2",
"proposal": "Amend the Rules of Procedure in Hungarian and English.",
"result": "not explicitly reported",
"individual_votes": [
{
"actor_id": "A3",
"vote": "abstain"
}
]
}

If a vote is mentioned but the result is not available, use:

**`"result": "not explicitly reported"`**

Do not infer the result from subsequent statements unless the transcript explicitly establishes it.

---

# 19. Final JSON schema

Return ONLY valid JSON.

Do not return Markdown.

Do not return explanations outside the JSON.

Use exactly this overall structure:

{
"contains_conversation": 0,
"contains_voting": 0,

"discussion_segments": [
{
"segment_id": "S1",
"title": "Short discussion title",
"location": "Approximate location",
"summary": "Concise summary",
"actor_ids": ["A1", "A2"],
"concept_ids": ["C1", "C2"]
}
],

"actors": [
{
"actor_id": "A1",
"actor": "Full name"
}
],

"concepts_discussed": [
{
"concept_id": "C1",
"concept": "Concept name",
"description": "Concise description",
"summary_of_positions": "Concise neutral summary of the positions expressed",
"discussion_segment_ids": ["S1"],

```
  "actor_attitudes": [
    {
      "actor_id": "A1",
      "attitude": "positive",
      "confidence": 0.95,
      "evidence": "Concise supporting evidence",
      "discussion_segment_ids": ["S1"]
    }
  ]
}

```

],

"attitude_matrix": {
"actors": [
{
"actor_id": "A1",
"actor": "Full name",
"attitudes": {
"C1": {
"attitude": "positive",
"confidence": 0.95,
"segment_ids": ["S1"]
},
"C2": {
"attitude": "not_discussed",
"confidence": null,
"segment_ids": []
}
}
}
]
},

"voting_events": [
{
"vote_id": "V1",
"subject": "Subject of vote",
"segment_id": "S1",
"proposal": "Proposal being voted on",
"result": "Result if explicitly stated, otherwise 'not explicitly reported'",
"individual_votes": [
{
"actor_id": "A1",
"vote": "yes | no | abstain | unclear"
}
]
}
]
}

---

# 20. Empty-value rules

If there is no substantive discussion:

- **`contains_conversation`** = **`0`**
- **`discussion_segments`** = **`[]`**
- **`concepts_discussed`** = **`[]`**
- **`attitude_matrix.actors`** = **`[]`**

If there is no voting:

- **`contains_voting`** = **`0`**
- **`voting_events`** = **`[]`**

If there is discussion but no identifiable individual actors:

- use the identifiable group/institution as the actor where appropriate.

If a concept was discussed but an actor's attitude cannot be determined:

- use **`"unclear"`**.

If an actor did not discuss a concept:

- use **`"not_discussed"`** in the matrix.

Never substitute **`"neutral"`** for **`"not_discussed"`**.

---

# 21. Final quality-control checks

Before returning the JSON, verify:

1. Every **`concept_id`** referenced by a discussion segment exists in **`concepts_discussed`**.
2. Every **`actor_id`** referenced anywhere exists in **`actors`**.
3. Every segment ID referenced by an actor-concept relationship exists in **`discussion_segments`**.
4. Every concept in the attitude matrix exists in **`concepts_discussed`**.
5. Every actor in the attitude matrix exists in **`actors`**.
6. Every matrix attitude is supported by a corresponding detailed actor-concept relationship.
7. **`"not_discussed"`** is used when there is no evidence of discussion.
8. **`"neutral"`** is used only when the actor actually discusses the concept without expressing a positive or negative attitude.
9. Every positive, negative, mixed, or unclear classification has supporting evidence.
10. Every confidence score is between **`0`** and **`1`**.
11. No external knowledge has been introduced.
12. No actor has been assigned an attitude merely because another actor expressed that attitude.
13. No voting result has been invented.
14. The JSON is syntactically valid.

## Text to analyse

—————

[INSERT TRANSCRIPT HERE]

—————

| Actor                       | Ethics Code / Ethics Committee rules | Senate Rules amendment | Equal Opportunities / GEP |               V4 conference postponement | Institutional political neutrality | Academic freedom / scientific integrity |
| --------------------------- | -----------------------------------: | ---------------------: | ------------------------: | ---------------------------------------: | ---------------------------------: | --------------------------------------: |
| **Bruno van Pottelsberghe** |                               🟢 Pro |                 🟢 Pro |       🟢 Pro / supportive |                                   🟢 Pro |                             🟢 Pro |                    🟡 Mixed / qualified |
| **Gábor Toronyai**          |                               🟢 Pro |        ⚪ Not discussed |           ⚪ Not discussed |                          ⚪ Not discussed |                    ⚪ Not discussed |                                  🟢 Pro |
| **Gábor Klimkó**            |                      ⚪ Not discussed | 🟡 Qualified / abstain |           ⚪ Not discussed |                          ⚪ Not discussed |                    ⚪ Not discussed |                            🟡 Qualified |
| **Judit Nagy**              |                      ⚪ Not discussed |        ⚪ Not discussed |                    🟢 Pro |                          ⚪ Not discussed |                    ⚪ Not discussed |                         ⚪ Not discussed |
| **Eszter Kirs**             |                      ⚪ Not discussed |        ⚪ Not discussed |           ⚪ Not discussed |                                🔴 Contra |                          🔴 Contra |                                  🟢 Pro |
| **Kata Apáti-Tóth**         |                      ⚪ Not discussed |        ⚪ Not discussed |           ⚪ Not discussed |           🟢 Pro / supports postponement |                             🟢 Pro |                            🟡 Qualified |
| **Ágnes Szunomár**          |                      ⚪ Not discussed |        ⚪ Not discussed |           ⚪ Not discussed | 🔴 Contra / concerned about cancellation |                           🟡 Mixed |                                  🟢 Pro |
| **Mánuel László Mágó**      |                      ⚪ Not discussed |        ⚪ Not discussed |           ⚪ Not discussed |                                🔴 Contra |                          🔴 Contra |                         🟢 Strongly pro |
| **András Rácz**             |                      ⚪ Not discussed |        ⚪ Not discussed |           ⚪ Not discussed |                       🟡 Mixed / unclear |                 🟡 Mixed / unclear |               🟢 Presumably supportive* |
| **Senate as a body**        |                               🟢 Pro |                 🟢 Pro |                    🟢 Pro |        ⚪ No collective position recorded |  ⚪ No collective position recorded |       ⚪ No collective position recorded |

## Optinal chunking

In [ ]:
import pandas as pd
from Helper import Log

In [ ]:
df = pd.read_csv('senate_meeting_minutes\output\senate_meeting_minutes_hun_202609231622.csv')

In [ ]:
import os
import io

MAX_BYTES = 1 * 1024 * 1024  # 1 MB

part_number = 1
start = 0
n = len(df)

while start < n:
    # Grow the chunk until the next row would push us over 1 MB
    end = start + 1
    while end <= n:
        candidate = df.iloc[start:end]
        # Measure the exact size the JSON will have on disk
        buffer = io.StringIO()
        candidate.to_json(
            buffer,
            orient="records",
            force_ascii=False,
            indent=2,
        )
        size = len(buffer.getvalue().encode("utf-8"))
        if size > MAX_BYTES and end > start + 1:
            # Last row made it too big → drop it
            end -= 1
            break
        end += 1

    chunk = df.iloc[start:end]
    with Log.save_file(f"{WORK_DIR}/json/senate_meeting_minutes_hun_part{part_number}.json") as filename:
        chunk.to_json(
            filename,
            orient="records",
            force_ascii=False,
            indent=2,
        )

    start = end
    part_number += 1